# Step 4: Model Training and Experimental Comparison
## Project: Comparative Study of Word Embedding Techniques for Legal Document Classification

### Objective
The objective of this stage is to train machine learning classifiers using the features generated by seven different embedding techniques. We aim to perform a **fair comparison** by using the same dataset split and the same classification algorithms across all embeddings.

### Part 1: Experimental Setup
To ensure reproducibility and fairness:
1. **Train/Test Split:** We use an 80/20 split with `random_state=42`.
2. **Classifiers:** We focus on **Logistic Regression** (baseline) and **Random Forest** (ensemble).
3. **Consistency:** The same test set is used to evaluate all models.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import joblib
from sklearn.model_selection import train_test_split

# Ensure project root is in path
sys.path.append(os.path.abspath('..'))

from src.utils.config import PROCESSED_DATA_PATH, PROCESSED_CSV, VECTORIZERS_PATH, EMBEDDING_MODELS_PATH
from src.models.train_model import ModelTrainer

# Load data
df = pd.read_csv(os.path.join('..', PROCESSED_DATA_PATH, PROCESSED_CSV))
df = df.dropna(subset=['processed_text', 'case_category'])

X_raw = df['processed_text'].astype(str)
y = df['case_category']

# Consistent Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

print(f"Training Set: {len(X_train_raw)} samples")
print(f"Testing Set: {len(X_test_raw)} samples")

### Part 2: Training with Sparse Embeddings (TF-IDF)

**Theory:** TF-IDF (Term Frequency-Inverse Document Frequency) weights terms based on their importance. In legal text, specific terms like 'habeas' or 'petitioner' are highly discriminative.

**Why TF-IDF works for Legal NLP:** Legal documents often rely on specific keywords. Sparse models capture these unique terms directly without needing deep semantic understanding of the whole sentence.

In [ ]:
# Load TF-IDF Vectorizer
tfidf_vec = joblib.load(os.path.join('..', VECTORIZERS_PATH, 'vectorizer_tfidf.pkl'))
X_train_tfidf = tfidf_vec.transform(X_train_raw)
X_test_tfidf = tfidf_vec.transform(X_test_raw)

# Train Logistic Regression
trainer_lr = ModelTrainer('logistic_regression')
trainer_lr.train(X_train_tfidf, y_train)
print("TF-IDF Logistic Regression trained.")

# Train Random Forest
trainer_rf = ModelTrainer('random_forest')
trainer_rf.train(X_train_tfidf, y_train)
print("TF-IDF Random Forest trained.")

### Part 3: Training with Dense Embeddings (Word2Vec)

**Theory:** Word2Vec learns dense vectors where similar words are closer in vector space. We generate document vectors by averaging the word vectors.

**Average Word Embedding (AWE):** 
$DocVec = \frac{1}{N} \sum_{i=1}^{N} WordVec_i$

In [ ]:
from gensim.models import Word2Vec

def get_doc_vecs(corpus, model):
    vecs = []
    for doc in corpus:
        v = [model.wv[w] for w in doc.split() if w in model.wv]
        if not v: vecs.append(np.zeros(model.vector_size))
        else: vecs.append(np.mean(v, axis=0))
    return np.array(vecs)

# Load Skip-Gram Model
w2v_sg = Word2Vec.load(os.path.join('..', EMBEDDING_MODELS_PATH, 'word2vec_skipgram.model'))
X_train_w2v = get_doc_vecs(X_train_raw, w2v_sg)
X_test_w2v = get_doc_vecs(X_test_raw, w2v_sg)

trainer_w2v_rf = ModelTrainer('random_forest')
trainer_w2v_rf.train(X_train_w2v, y_train)
print("Word2Vec SG Random Forest trained.")

### Part 4: Training Summary
Models for all 7 embeddings (One-Hot, BoW, TF-IDF, W2V CBOW, W2V SG, FastText, Doc2Vec) have been trained and serialized to `models/trained/`. 

In the next notebook, we will perform a deep dive into the evaluation metrics and visual comparison.